In [4]:
!pip install git+https://github.com/songlab-cal/gpn.git

  Cloning https://github.com/songlab-cal/gpn.git to /tmp/pip-req-build-8s2hyn8s
  ERROR: Error [Errno 2] No such file or directory: 'git' while executing command git version
ERROR: Cannot find command 'git' - do you have 'git' installed and in your PATH?


In [1]:
import csv
import os

import gpn.star.model  # MUST import to register architecture with AutoModel
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from transformers import AutoConfig, AutoModelForMaskedLM, Trainer, TrainingArguments
import tempfile
from tqdm import tqdm

from gpn.star.data import GenomeMSA
from gpn.star.vep import VEPInference, MLMforVEPModel

# Fix phylo_dist_path to use the copy bundled inside the model folder,
# since the original path in config.json is a relative training-time path.
_orig_mlm_init = MLMforVEPModel.__init__

def _patched_init(self, model_path):
    torch.nn.Module.__init__(self)
    config = AutoConfig.from_pretrained(model_path)
    if not os.path.exists(config.phylo_dist_path):
        config.phylo_dist_path = os.path.join(model_path, "phylo_dist")
    self.model = AutoModelForMaskedLM.from_pretrained(model_path, config=config)
    self.model.eval()

MLMforVEPModel.__init__ = _patched_init


/gpfs/home/pl2948/.conda/envs/gpn/lib/python3.10/site-packages/requests/__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [2]:
# ──────────────────────────────────────────────
# Config  ← EDIT THESE
# ──────────────────────────────────────────────
VARIANTS_CSV = "/gpfs/home/pl2948/VEP_eval/ClinVarBenchmark_PB_202602.csv"
OUTPUT_DIR   = "/gpfs/home/pl2948/VEP_eval/gpnstar_chunks"

# Vertebrate model (100-way, 200M params)
MODEL_PATH   = "/gpfs/scratch/pl2948/VEP_eval/gpn-star-hg38-v100-200m"

# Direct path to the decompressed zarr store.
# Structure after unpigz < 99.zarr.tar.gz | tar -x:
#   multiz100way/99.zarr/{1,2,...,22,X,Y}/   (per-chrom groups)
# 99 = number of species in this alignment.
MSA_ZARR_PATH = "/gpfs/scratch/pl2948/VEP_eval/multiz100way/99.zarr"
MSA_N_SPECIES = 99

WINDOW_SIZE  = 128   # vertebrate model context size (128 bp)

SCORE_COLS   = ["idx", "#CHROM", "POS", "REF", "ALT", "gpnstar_v_llr"]

In [3]:
# ──────────────────────────────────────────────
# Helpers
# ──────────────────────────────────────────────

def load_chunk(chunk_idx, n_chunks):
    df      = pd.read_csv(VARIANTS_CSV, low_memory=False)
    indices = np.array_split(np.arange(len(df)), n_chunks)[chunk_idx]
    return df.iloc[indices], indices


def build_hf_dataset(chunk_df):
    """
    VEPInference.tokenize_function() expects a HuggingFace Dataset with columns:
        chrom (str), pos (int, 1-based), ref (str), alt (str)
    """
    records = []
    for idx, row in chunk_df.iterrows():
        chrom = str(row["#CHROM"])
        # MSA zarr uses no "chr" prefix (e.g. "1", "2", "X")
        if chrom.startswith("chr"):
            chrom = chrom[3:]
        ref = str(row["REF"]).upper()
        alt = str(row["ALT"]).upper()
        # Skip non-SNVs (the model only supports single-nucleotide substitutions)
        if len(ref) != 1 or len(alt) != 1:
            continue
        if ref not in "ACGT" or alt not in "ACGT":
            continue
        records.append({
            "_orig_idx": idx,
            "chrom":     chrom,
            "pos":       int(row["POS"]),   # keep 1-based
            "ref":       ref,
            "alt":       alt,
            "#CHROM":    str(row["#CHROM"]),
            "POS":       int(row["POS"]),
            "REF":       ref,
            "ALT":       alt,
        })
    return pd.DataFrame(records)


# ──────────────────────────────────────────────
# Main
# ──────────────────────────────────────────────

def run_chunk(chunk_idx, n_chunks, per_device_batch_size=8, dataloader_num_workers=0):
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    out_path = os.path.join(OUTPUT_DIR, f"gpnstar_chunk{chunk_idx}.csv")

    # ── Resume: load already-completed idx ──
    done_idxs = set()
    if os.path.exists(out_path):
        try:
            done_df   = pd.read_csv(out_path, usecols=["idx"])
            done_idxs = set(done_df["idx"].tolist())
            print(f"[Resume] {len(done_idxs)} variants already done, skipping.")
        except Exception:
            print("[Resume] Could not read existing file, starting fresh.")

    file_is_new = not os.path.exists(out_path) or os.path.getsize(out_path) == 0
    csv_file = open(out_path, "a", newline="", buffering=1)
    writer   = csv.DictWriter(csv_file, fieldnames=SCORE_COLS)
    if file_is_new:
        writer.writeheader()

    # ── Load chunk ──
    chunk_df, indices = load_chunk(chunk_idx, n_chunks)
    print(f"\n=== Chunk {chunk_idx}/{n_chunks} | {len(chunk_df)} variants ===")

    records_df = build_hf_dataset(chunk_df)
    if records_df.empty:
        print("No valid SNVs in this chunk, exiting.")
        csv_file.close()
        return

    # Filter already-done
    records_df = records_df[~records_df["_orig_idx"].isin(done_idxs)]
    if records_df.empty:
        print("All variants already scored.")
        csv_file.close()
        return

    # ── Load MSA ──
    # subset_chroms speeds up loading — only load chromosomes we actually need
    subset_chroms = records_df["chrom"].unique().tolist()
    print(f"Loading MSA from {MSA_ZARR_PATH} (chroms: {subset_chroms}) ...")
    genome_msa_list = [
        GenomeMSA(
            MSA_ZARR_PATH,
            n_species=MSA_N_SPECIES,
            subset_chroms=subset_chroms,
            in_memory=False,
        )
    ]
    print("MSA loaded.")

    # ── Build VEPInference (holds model + tokenize logic) ──
    print(f"Loading model from {MODEL_PATH} ...")
    inference = VEPInference(
        MODEL_PATH,
        genome_msa_list,
        WINDOW_SIZE,
        disable_aux_features=False,
    )

    # ── Run inference via HuggingFace Trainer (same as official inference.py) ──
    hf_dataset = Dataset.from_pandas(
        records_df[["chrom", "pos", "ref", "alt"]].reset_index(drop=True)
    )
    hf_dataset.set_transform(inference.tokenize_function)

    training_args = TrainingArguments(
        output_dir=tempfile.TemporaryDirectory().name,
        per_device_eval_batch_size=per_device_batch_size,
        dataloader_num_workers=dataloader_num_workers,
        remove_unused_columns=False,
        torch_compile=False,   # set True if your torch version supports it
        fp16=True,
    )
    trainer = Trainer(model=inference.model, args=training_args)
    predictions = trainer.predict(test_dataset=hf_dataset).predictions
    # predictions is a 1-D numpy array of LLR scores, one per variant

    # ── Write results ──
    records_df = records_df.reset_index(drop=True)
    for i, llr in enumerate(predictions):
        row = records_df.iloc[i]
        writer.writerow({
            "idx":           int(row["_orig_idx"]),
            "#CHROM":        str(row["#CHROM"]),
            "POS":           int(row["POS"]),
            "REF":           str(row["REF"]),
            "ALT":           str(row["ALT"]),
            "gpnstar_v_llr": float(llr),
        })

    csv_file.close()
    print(f"Done. {len(predictions)} scores written to {out_path}")

In [4]:
run_chunk(0, 1)


=== Chunk 0/1 | 242132 variants ===
Loading MSA from /gpfs/scratch/pl2948/VEP_eval/multiz100way/99.zarr (chroms: ['1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '20', '21', '22', '3', '4', '5', '6', '7', '8', '9', 'X', 'Y']) ...
Loading MSA...
Loading MSA... Done
MSA loaded.
Loading model from /gpfs/scratch/pl2948/VEP_eval/gpn-star-hg38-v100-200m ...


/gpfs/home/pl2948/.conda/envs/gpn/lib/python3.10/contextlib.py:103: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


ref genome and variant file unmatched: [2, 3, 3, 2, 1, 1, 4, 3, 3, 3, 1, 2, 4, 3, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [2, 3, 3, 2, 1, 1, 4, 3, 3, 3, 1, 2, 4, 3, 2, 2, 3, 1, 3, 3, 2, 3, 4, 2, 3, 3, 2, 3, 2, 2, 3, 2]
ref genome and variant file unmatched: [3, 2, 2, 3, 4, 4, 1, 2, 2, 2, 4, 3, 1, 2, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [3, 2, 2, 3, 4, 4, 1, 2, 2, 2, 4, 3, 1, 2, 3, 3, 2, 4, 2, 2, 3, 2, 1, 3, 2, 2, 3, 2, 3, 3, 2, 3]
ref genome and variant file unmatched: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [3, 3, 2, 3, 2, 2, 2, 1, 3, 3, 3, 4, 1, 1, 3, 1, 4, 3, 1, 2, 3, 2, 2, 3, 3, 2, 3, 4, 2, 4, 4, 3]
ref genome and variant file unmatched: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [2, 2, 3, 2, 3, 3, 3, 4, 2, 2, 2, 1, 4, 4, 2, 4, 1, 2, 4, 3, 2, 3, 3, 2, 2, 3, 2, 1, 3, 1, 1, 2]
ref genome and variant file unmatched: [0, 0, 0, 0, 0, 0, 0, 0, 

In [5]:
import pandas as pd

version = 'updated'# "balancedgenome" #"balanced" # 'updated' # 2023 #

data = pd.read_csv(f"./clinvar_{version}.csv", low_memory=False)
data

,#CHROM,POS,ID,REF,ALT,ClinVar_label,ClinVar_gold_stars,ClinVarName_refseq_ids,ClinVarName_AAPOS,ClinVarName_AAREF,...,AlphaMissense,PhyloGPN,Evo2_40B,Rule_based,ntv3_pre_position_llr,ntv3_pre_seq_pllr,ntv3_post_position_llr,ntv3_post_seq_pllr,ntv3_post_log2fc_max,vesm_score
0,1,930165,1164676,G,A,0.0,1.0,NM_001385641.1,207.0,R,...,NaN,-1.801111,-0.001966,0.357648,-1.033955,-1.668930e-06,-1.666055,0.000043,2.827893,NaN
1,1,930204,1170208,G,A,0.0,2.0,NM_001385641.1,220.0,R,...,NaN,-0.810768,-0.001260,0.357648,-1.569786,4.708767e-06,-1.588054,0.000701,3.240415,NaN
2,1,930285,1165489,G,A,0.0,1.0,NM_001385641.1,247.0,R,...,NaN,-0.815928,0.000659,0.357648,-1.588131,3.576279e-07,-1.680781,0.000585,3.104318,NaN
3,1,930314,1170010,C,T,0.0,2.0,NM_001385641.1,257.0,H,...,NaN,-1.633337,-0.003418,0.357648,-3.150200,7.659197e-06,-3.216790,-0.000197,1.822163,NaN
4,1,930325,1168187,C,T,0.0,1.0,NM_001385641.1,260.0,I,...,NaN,-1.464217,-0.000417,0.005252,-4.653107,-1.162291e-05,-3.354657,0.000510,2.667778,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
242127,Y,56960499,726033,A,G,0.0,1.0,NM_001304990.2,36.0,K,...,NaN,-0.414166,-0.000091,0.005252,-1.494453,-5.513430e-06,-2.435236,-0.001451,2.540560,NaN
242128,Y,57084754,777848,C,G,0.0,2.0,NM_005638.6,109.0,V,...,NaN,-1.074717,-0.000446,0.005252,-1.905708,-1.013279e-06,-1.233571,-0.002534,2.924839,NaN
242129,Y,57087054,721652,T,C,0.0,2.0,NM_005638.6,127.0,M,...,0.08,-3.095034,-0.000956,0.357648,-6.131863,-4.357100e-05,-2.936291,-0.000638,2.801300,-3.9775
242130,Y,57190090,785714,G,A,0.0,1.0,NM_002186.3,NaN,NaN,...,NaN,-0.255677,-0.000129,0.020306,-2.886108,5.185604e-06,0.014036,-0.005069,2.687465,NaN


In [8]:
gpn = pd.read_csv('/gpfs/home/pl2948/VEP_eval/gpnstar_chunks/gpnstar_chunk0.csv', low_memory=False)
gpn

,idx,#CHROM,POS,REF,ALT,gpnstar_v_llr
0,0,1,930165,G,A,-5.484859
1,1,1,930204,G,A,-3.155202
2,2,1,930285,G,A,-1.901591
3,3,1,930314,C,T,-4.783436
4,4,1,930325,C,T,-4.984299
...,...,...,...,...,...,...
242127,242127,Y,56960499,A,G,-2.214205
242128,242128,Y,57084754,C,G,-4.122275
242129,242129,Y,57087054,T,C,-3.547984
242130,242130,Y,57190090,G,A,-2.359769


In [10]:
key_cols = ["#CHROM", "POS", "REF", "ALT"]
keys_match = (data[key_cols].values == gpn[key_cols].values).all()
print(f"Keys match efore sorting: {keys_match}")

Keys match efore sorting: True


In [11]:
data['gpnstar_v_llr'] = gpn['gpnstar_v_llr']

data.to_csv("./clinvar_updated.csv", index=False)